In [4]:
import pandas as pd
import geopandas as gpd
import topojson as tp

In [5]:
latest_df = pd.read_csv(
    "../../reports/deidentified_overdose_201201202408_zips_0311.csv"
)

/tmp/ipykernel_2545121/720356717.py:1: DtypeWarning: Columns (12,29) have mixed types. Specify dtype option on import or set low_memory=False.
  latest_df = pd.read_csv(


In [6]:
latest_df

,Unnamed: 0,CaseNumber,Age,Gender,Race,ResidenceType,DeathPlace,DeathAddress,DeathCity,EventPlace,...,LABEL,ShapeSTArea,ShapeSTLength,ZIPCODE,Shape_Length,Shape_Area,MonthYear,Year,YearWeek,Age_Bin
0,0,2012-00007,56.0,Male,LATINE,NaN,SIDEWALK,F/O 1052 SANFORD AVE,WILMINGTON,NaN,...,2946.20,9.059969e+06,12685.368502,90744,92694.910150,2.432981e+08,2012-01,2012,2012-01,50-59
1,1,2012-00017,51.0,Male,ASIAN,NaN,DRIVEWAY,20427 CLARKDALE AVE,LAKEWOOD,DRIVEWAY,...,5551.05,1.184700e+07,18475.845945,90715,41993.949445,4.778823e+07,2012-01,2012,2012-01,50-59
2,2,2012-00018,50.0,Male,LATINE,NaN,HOSPITAL,309 WEST BEVERLY BLVD.,MONTEBELLO,NaN,...,5301.01,1.089460e+07,14077.604472,90640,104064.388983,2.370969e+08,2012-01,2012,2012-01,50-59
3,3,2012-00071,47.0,Male,LATINE,HOME,SIDEWALK,5920 S. CENTRAL AVE.,LOS ANGELES,NaN,...,5328.00,5.931393e+06,10039.079857,90001,48677.278478,9.556340e+07,2012-01,2012,2012-01,40-49
4,4,2012-00089,30.0,Male,LATINE,HOME,RESIDENCE,5026 1/2 ECHO STREET,LOS ANGELES,RESIDENCE,...,1838.20,3.322203e+06,10487.572705,90042,69418.508152,1.260515e+08,2012-01,2012,2012-01,30-39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19915,19940,2024-13760,48.0,Male,LATINE,NaN,Construction\nsite,1808 South\nSaint Andrews\nPlace,Los Angeles,NaN,...,2213.02,4.974449e+06,10183.103404,90019,66607.662628,1.067585e+08,2024-08,2024,2024-34,40-49
19916,19941,2024-13819,30.0,Female,LATINE,NaN,Private\nResidence,4941 Passons\nBoulevard,Pico Rivera,Private\nResidence,...,5004.02,1.437915e+07,15368.244441,90660,105083.371956,2.648887e+08,2024-08,2024,2024-34,30-39
19917,19942,2024-13820,46.0,Female,WHITE,NaN,Residence,292 Gaviota\nAvenue,Long Beach,Residence,...,5765.03,3.356438e+06,7911.514399,90802,181881.275506,1.684347e+08,2024-08,2024,2024-34,40-49
19918,19943,2024-13833,48.0,Male,LATINE,NaN,PRISON,44750 60th\nstreet west,Lancaster,PRISON,...,9010.03,2.803597e+07,21202.399755,93536,630966.131352,7.768524e+09,2024-08,2024,2024-34,40-49


In [11]:
import pandas as pd
import geopandas as gpd
import topojson as tp

# 1. Load your CSV which has ZIPCODE, Year, and drug columns
latest_df = pd.read_csv(
    "../../reports/deidentified_overdose_201201202408_zips_0311.csv"
)

drug_cols = [
    "Methamphetamine",
    "Heroin",
    "Cocaine",
    "Fentanyl",
    "Alcohol",
    "Prescription.opioids",
    "Any Opioids",
    "Benzodiazepines",
    "Others",
    "Any Drugs",
]

# Include 'Year' in the subset if you want per-year counts
df_subset = latest_df[["ZIPCODE", "Year"] + drug_cols].copy()

# 2. Aggregate overdose counts by ZIPCODE AND Year
df_agg = df_subset.groupby(["ZIPCODE", "Year"])[drug_cols].sum().reset_index()

# Rename columns for clarity
df_agg.columns = ["ZIPCODE", "Year"] + [f"{col}_Count" for col in drug_cols]

# 3. Load your ZIP code geometry
zip_gdf = gpd.read_file("../../data/zipcodes.geojson")
zip_gdf["ZIPCODE"] = zip_gdf["ZIPCODE"].astype(str)
df_agg["ZIPCODE"] = df_agg["ZIPCODE"].astype(str)

# 4. Merge aggregated overdose counts with ZIPCODE geometry
#    This will create multiple rows per ZIPCODE if multiple Years exist.
zip_overdose_gdf = df_agg.merge(zip_gdf, on="ZIPCODE", how="left")

# 5. Melt to create a single Overdose_Type column (long format)
new_drug_cols = [f"{col}_Count" for col in drug_cols]

df_long = zip_overdose_gdf.melt(
    id_vars=[
        "ZIPCODE",
        "Year",
        "OBJECTID",  # if it exists in your geojson
        "Shape_Length",  # if it exists
        "Shape_Area",  # if it exists
        "geometry",
    ],
    value_vars=new_drug_cols,
    var_name="Overdose_Type",
    value_name="Overdose_Count",
)

df_long_gdf = gpd.GeoDataFrame(
    df_long,
    geometry="geometry",
)

# 6. Convert CRS to EPSG:3857 before TopoSimplify
df_long_3857 = df_long_gdf.to_crs(epsg=3857)

# 7. Create a Topology and simplify
topo = tp.Topology(df_long_3857, prequantize=False)
simple = topo.toposimplify(0.01).to_gdf()


# Save final GeoDataFrame
simple.to_file("zip_overdose_all_year_drug_long.gpkg", driver="GPKG")

/tmp/ipykernel_2545121/4043067860.py:6: DtypeWarning: Columns (12,29) have mixed types. Specify dtype option on import or set low_memory=False.
  latest_df = pd.read_csv(


KeyboardInterrupt: 

In [12]:
df_long_3857

,ZIPCODE,Year,OBJECTID,Shape_Length,Shape_Area,geometry,Overdose_Type,Overdose_Count
0,90001,2012,1,48677.278478,9.556340e+07,"POLYGON ((-13162792.911 4027356.791, -13162803...",Methamphetamine_Count,0
1,90001,2013,1,48677.278478,9.556340e+07,"POLYGON ((-13162792.911 4027356.791, -13162803...",Methamphetamine_Count,2
2,90001,2015,1,48677.278478,9.556340e+07,"POLYGON ((-13162792.911 4027356.791, -13162803...",Methamphetamine_Count,0
3,90001,2016,1,48677.278478,9.556340e+07,"POLYGON ((-13162792.911 4027356.791, -13162803...",Methamphetamine_Count,1
4,90001,2017,1,48677.278478,9.556340e+07,"POLYGON ((-13162792.911 4027356.791, -13162803...",Methamphetamine_Count,0
...,...,...,...,...,...,...,...,...
31665,93591,2018,308,426758.841648,2.604198e+09,"POLYGON ((-13127249.894 4117873.247, -13127270...",Any Drugs_Count,1
31666,93591,2021,308,426758.841648,2.604198e+09,"POLYGON ((-13127249.894 4117873.247, -13127270...",Any Drugs_Count,1
31667,93591,2022,308,426758.841648,2.604198e+09,"POLYGON ((-13127249.894 4117873.247, -13127270...",Any Drugs_Count,1
31668,93591,2023,308,426758.841648,2.604198e+09,"POLYGON ((-13127249.894 4117873.247, -13127270...",Any Drugs_Count,2


In [3]:
latest_df = latest_df.drop(columns=["Unnamed: 0"])

In [4]:
drug_cols = [
    "Methamphetamine",
    "Heroin",
    "Cocaine",
    "Fentanyl",
    "Alcohol",
    "Prescription.opioids",
    "Any Opioids",
    "Benzodiazepines",
    "Others",
    "Any Drugs",
]

In [5]:
# Keep ZIPCODE and drug columns only
df_subset = latest_df[["ZIPCODE"] + drug_cols].copy()

# Aggregate overdose counts by ZIPCODE
df_agg = df_subset.groupby(["ZIPCODE"])[drug_cols].sum().reset_index()

# Rename columns for clarity
df_agg.columns = ["ZIPCODE"] + [f"{col}_Count" for col in drug_cols]

In [6]:
zip_gdf = gpd.read_file("../../data/zipcodes.geojson")
zip_gdf["ZIPCODE"] = zip_gdf["ZIPCODE"].astype(str)
df_agg["ZIPCODE"] = df_agg["ZIPCODE"].astype(str)

# Merge aggregated overdose counts with ZIPCODE geometry
zip_overdose_gdf = zip_gdf.merge(df_agg, on="ZIPCODE", how="left")

In [7]:
new_drug_cols = [f"{col}_Count" for col in drug_cols]

df_long = zip_overdose_gdf.melt(
    id_vars=[
        "ZIPCODE",
        "OBJECTID",
        "Shape_Length",
        "Shape_Area",
        "geometry",
    ],  # or other common fields
    value_vars=new_drug_cols,
    var_name="Overdose_Type",
    value_name="Overdose_Count",
)

In [13]:
topo = tp.Topology(df_long.to_crs({"init": "epsg:3857"}), prequantize=False)
simple = topo.toposimplify(0.01).to_gdf()

/home/afunnell/miniconda3/envs/arcgis/lib/python3.11/site-packages/pyproj/crs/crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)


In [15]:
simple.to_file(f"../../reports/datafordash/all_years_long_0317_simplified.gpkg")

In [16]:
df_long

,ZIPCODE,OBJECTID,Shape_Length,Shape_Area,geometry,Overdose_Type,Overdose_Count
0,90001,1,48677.278478,9.556340e+07,"POLYGON ((-118.24338 33.98924, -118.24348 33.9...",Methamphetamine_Count,42.0
1,90002,2,44972.017035,8.275222e+07,"POLYGON ((-118.23431 33.96101, -118.23442 33.9...",Methamphetamine_Count,17.0
2,90003,3,63303.481758,1.026431e+08,"POLYGON ((-118.28285 33.97597, -118.28285 33.9...",Methamphetamine_Count,75.0
3,90004,4,52471.897635,8.395697e+07,"POLYGON ((-118.2841 34.08349, -118.28438 34.08...",Methamphetamine_Count,78.0
4,90005,5,88978.011128,3.688581e+07,"MULTIPOLYGON (((-118.33541 34.06179, -118.3353...",Methamphetamine_Count,53.0
...,...,...,...,...,...,...,...
3125,90090,316,7248.507956,6.841331e+05,"POLYGON ((-118.24815 34.07238, -118.24688 34.0...",Any Drugs_Count,NaN
3126,91709,317,13364.561099,3.788000e+06,"POLYGON ((-117.7471 34.02073, -117.74711 34.02...",Any Drugs_Count,NaN
3127,91748,318,105409.004195,4.055949e+08,"POLYGON ((-117.8631 33.94627, -117.86544 33.94...",Any Drugs_Count,29.0
3128,91765,319,135366.724864,5.230031e+08,"POLYGON ((-117.79786 34.04021, -117.79813 34.0...",Any Drugs_Count,20.0


In [ ]:
df_long

In [ ]:
df_long.to_file("zip_overdose_dashboard0314long.gpkg")

### Creating each subset for all years

In [ ]:
zip_gdf = gpd.read_file("../../data/zipcodes.geojson")

In [ ]:
year_dfs = {}
long_year_dfs = {}
for year in latest_df["Year"].unique():
    # Keep ZIPCODE and drug columns only

    year_subset = latest_df[latest_df["Year"] == year]

    year_subset = year_subset[["ZIPCODE"] + drug_cols].copy()

    df_agg = year_subset.groupby(["ZIPCODE"])[drug_cols].sum().reset_index()

    df_agg.columns = ["ZIPCODE"] + [f"{col}_Count" for col in drug_cols]

    zip_gdf["ZIPCODE"] = zip_gdf["ZIPCODE"].astype(str)
    df_agg["ZIPCODE"] = df_agg["ZIPCODE"].astype(str)

    # Merge aggregated overdose counts with ZIPCODE geometry
    zip_overdose_gdf_year = zip_gdf.merge(df_agg, on="ZIPCODE", how="left")

    year_dfs[year] = zip_overdose_gdf_year

    new_drug_cols = [f"{col}_Count" for col in drug_cols]

    df_long = zip_overdose_gdf_year.melt(
        id_vars=[
            "ZIPCODE",
            "OBJECTID",
            "Shape_Length",
            "Shape_Area",
            "geometry",
        ],  # or other common fields
        value_vars=new_drug_cols,
        var_name="Overdose_Type",
        value_name="Overdose_Count",
    )

    df_long["Overdose_Count"] = df_long["Overdose_Count"].fillna(0)

    long_year_dfs[year] = df_long

    topo = tp.Topology(df_long.to_crs({"init": "epsg:3857"}), prequantize=False)
    simple = topo.toposimplify(1).to_gdf()

    simple.to_file(f"../../reports/datafordash/{year}_long_0317_simplified.gpkg")

In [ ]:
year_dfs[2024]